# Module 15 — Agentic Workflows Without Chaos

Companion notebook to [`docs/modules/15_agentic_workflows_without_chaos.md`](../docs/modules/15_agentic_workflows_without_chaos.md).

Builds both shapes from the theory doc's preferred mental model, for real, on the same task
("how many open tickets are there?" over a real SQLite fixture database): a ReAct loop (the
"avoid" shape) that a real adversarial prompt can break, and a deterministic `WorkflowGraph`
(the "prefer" shape) that's immune to the same prompt by construction. Every planning/execution
mechanism is deterministic Python; the two LLM-dependent pieces (ReAct's step reasoning, one
bounded workflow decision point) are `FakeRuntime`/`ScriptedTurnRuntime`-backed.


In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "packages"))
sys.path.insert(0, str(REPO_ROOT / "scripts" / "module_15"))
print(f"Repo root: {REPO_ROOT}")


Repo root: /Users/bhakti/workspace/local_ai_app


## 1. Safety budget and loop prevention — real, deterministic guardrails

In [2]:
from local_ai_agents.planners.loop_prevention import LoopDetectedError, LoopGuard
from local_ai_agents.planners.safety_budget import AgentSafetyBudget, SafetyBudgetExceededError

budget = AgentSafetyBudget(max_steps=2, max_tool_calls=5, max_runtime_seconds=60, max_tokens_total=8000)
budget.record_step()
budget.record_step()
try:
    budget.record_step()
except SafetyBudgetExceededError as exc:
    print("Safety budget stopped a 3rd step:", exc)

guard = LoopGuard(max_repeats=3)
for i in range(3):
    try:
        guard.record("sql_query", {"query": "SELECT 1"})
        print(f"call {i + 1}: allowed")
    except LoopDetectedError as exc:
        print(f"call {i + 1}: LoopGuard tripped -> {exc}")


Safety budget stopped a 3rd step: max_steps exceeded (2)
call 1: allowed
call 2: allowed
call 3: LoopGuard tripped -> Tool 'sql_query' proposed with identical arguments 3 times in a row


## 2. Lab 1-2: ReAct loop, then broken by a real adversarial prompt

The "avoid" shape from the preferred mental model, implemented for real - and then genuinely
broken, not just described as fragile.

In [3]:
from react_loop_demo import run_lab as run_react_lab, result_to_markdown as react_to_markdown

react_result = await run_react_lab()
print(react_to_markdown(react_result))


# Labs 1-2 - ReAct loop, then broken by an adversarial prompt

- Request: How many open tickets are there?
## Lab 1: happy path
- Stopped reason: final_answer
- Final answer: There are 3 open tickets.
- Memory entries: 3
## Lab 2: adversarial break
- Stopped reason: loop_detected
- LLM calls made before LoopGuard tripped: 3



## 3. Labs 3-4: the same task as a deterministic workflow graph — immune by construction

Not "loop prevention catches it eventually" - there is no step where the model chooses a tool or
repeats an action at all. Also demonstrates a real human-approval interrupt on a dangerous node.

In [4]:
from workflow_graph_demo import run_lab as run_graph_lab, result_to_markdown as graph_to_markdown

graph_result = await run_graph_lab()
print(graph_to_markdown(graph_result))


# Labs 3-4 - deterministic workflow graph, immune by construction, with approval interrupt

- Request: How many open tickets are there?
- Dangerous node denied by default (no approval gate): approval_denied
- Dangerous node approved: end -> There are 3 open tickets requiring attention.
- summary.txt actually written to disk: True
- Same adversarial runtime from Lab 2, run against the graph: end (only 1 LLM call(s) made - no loop possible)



## 4. Lab 5: checkpointing and resume across a real restart

A genuinely new `WorkflowExecutor` and `CheckpointStore`, pointed at the same SQLite file after
the first run failed partway through - not a mock, an actual file read from disk.

In [5]:
from checkpoint_demo import run_lab as run_checkpoint_lab, result_to_markdown as checkpoint_to_markdown

checkpoint_result = await run_checkpoint_lab()
print(checkpoint_to_markdown(checkpoint_result))


# Lab 5 - checkpointing and resume across a real restart

- First run stopped at 'categorize' (failed) with state {'tickets_fetched': 5}
- Resumed run (new executor, new store, same SQLite file) result: end
- Final state after resume: {'tickets_fetched': 5, 'categorized': True, 'done': True}
- 'fetch' was not re-run - its result (tickets_fetched=5) survived into the resumed state.



## 5. Lab 6: agent task success evaluation

A golden set scored by exact match on final workflow state - including one deliberately wrong
expectation, to prove the scorer actually discriminates rather than rubber-stamping every case.

In [6]:
from evaluate_task_success import run_lab as run_eval_lab, result_to_markdown as eval_to_markdown

eval_rows = await run_eval_lab()
print(eval_to_markdown(eval_rows))


# Lab 6 - agent task success evaluation

- Task success rate: 2/3

- normal_run: success=True, stopped_reason=end, open_ticket_count=3
- verbose_summary: success=True, stopped_reason=end, open_ticket_count=3
- deliberately_wrong_expectation: success=False, stopped_reason=end, open_ticket_count=3



## 6. State machine == graph with unambiguous branches — the same engine, two topics

`WorkflowGraph` implements both "state machine agents" and "graph-based agents" (theory doc
§4-5) as one engine: a state machine is simply a graph whose conditional edges never have more
than one branch that's actually true for a given state.

In [7]:
from local_ai_agents.planners.workflow_graph import WorkflowGraph, WorkflowNode

async def noop(state, memory):
    return state

graph = WorkflowGraph(start_node="check")
graph.add_node(WorkflowNode(name="check", run=noop))
graph.add_node(WorkflowNode(name="approved_path", run=noop))
graph.add_node(WorkflowNode(name="denied_path", run=noop))
graph.add_edge("check", "approved_path", condition=lambda s: s.get("approved") is True)
graph.add_edge("check", "denied_path", condition=lambda s: True)  # fallback branch

print("approved=True  ->", graph.next_node("check", {"approved": True}))
print("approved=False ->", graph.next_node("check", {"approved": False}))


approved=True  -> approved_path
approved=False -> denied_path
